# Practical Exercise 3: Uniform Cost Search for the Self-Driving Trailer

**Group 9**

| Admission No. | Name |
|---|---|
| 184540 | Onwenga Francis Bosire |
| 192077 | Ogero Mark Otinga |
| 189720 | Kirwa Kipchumba Brian |

**Problem:** Using the directed graph and path costs from Part 1, find the cheapest route for the trailer from **Nairobi** (initial state) to **Kisumu** (goal state) using Uniform Cost Search (UCS).

## 1. Road network and path costs

Each road is stored as an edge with its distance in km. Roads are bidirectional, so each edge is added in both directions.

In [ ]:
roads = [
    ("Nairobi",   "Naivasha",  76.60),
    ("Naivasha",  "Gilgil",    26.90),
    ("Gilgil",    "Nakuru",    36.77),
    ("Nakuru",    "Kericho",   88.70),
    ("Kericho",   "Kisumu",    65.83),
    ("Nairobi",   "Thika",     38.96),
    ("Thika",     "Nyeri",     70.83),
    ("Nyeri",     "Nyahururu", 82.68),
    ("Nyahururu", "Nakuru",    48.39),
    ("Nakuru",    "Eldoret",  126.15),
    ("Eldoret",   "Kisumu",    89.96),
]

graph = {}
for a, b, d in roads:
    graph.setdefault(a, []).append((b, d))
    graph.setdefault(b, []).append((a, d))

for town, nbrs in graph.items():
    print(f"{town:10s} -> {nbrs}")

: 

## 2. Uniform Cost Search

UCS keeps a **priority queue (frontier)** ordered by the path cost `g(n)` from the start. It always expands the cheapest node first, so the first time the goal is removed from the frontier, the path found is optimal.

The function prints each expansion (node, cost so far, frontier) to show the traversal as done in class.

In [ ]:
import heapq

def uniform_cost_search(graph, start, goal, verbose=True):
    # frontier entries: (path_cost, tie_breaker, town, path)
    counter = 0
    frontier = [(0.0, counter, start, [start])]
    explored = set()
    best_cost = {start: 0.0}
    step = 0

    while frontier:
        cost, _, town, path = heapq.heappop(frontier)

        if town in explored:          # stale, more expensive entry
            continue

        step += 1
        if verbose:
            print(f"Step {step}: expand {town} (g = {cost:.2f} km)")

        if town == goal:              # goal test on expansion => optimal
            return path, cost, explored | {town}

        explored.add(town)

        for nbr, d in graph[town]:
            new_cost = cost + d
            if nbr not in explored and new_cost < best_cost.get(nbr, float("inf")):
                best_cost[nbr] = new_cost
                counter += 1
                heapq.heappush(frontier, (new_cost, counter, nbr, path + [nbr]))

        if verbose:   # show the live frontier (stale duplicates hidden)
            live = sorted((round(c, 2), t) for c, _, t, _ in frontier
                          if t not in explored and c == best_cost[t])
            print(f"         frontier: {live}")

    return None, float("inf"), explored

## 3. Run the search: Nairobi to Kisumu

In [ ]:
path, cost, explored = uniform_cost_search(graph, "Nairobi", "Kisumu")

print()
print("Optimal path :", " -> ".join(path))
print(f"Total cost   : {cost:.2f} km")
print("Towns expanded:", len(explored))

## 4. Verification against the Part 1 route costs

The four candidate routes from Part 1 are recomputed from the same road data. UCS should return the cheapest one.

In [ ]:
routes = {
    "Route 1": ["Nairobi","Naivasha","Gilgil","Nakuru","Kericho","Kisumu"],
    "Route 2": ["Nairobi","Thika","Nyeri","Nyahururu","Nakuru","Kericho","Kisumu"],
    "Route 3": ["Nairobi","Naivasha","Gilgil","Nakuru","Eldoret","Kisumu"],
    "Route 4": ["Nairobi","Thika","Nyeri","Nyahururu","Nakuru","Eldoret","Kisumu"],
}
dist = {(a, b): d for a, b, d in roads}
dist.update({(b, a): d for a, b, d in roads})

def path_cost(p):
    return sum(dist[(p[i], p[i+1])] for i in range(len(p) - 1))

for name, p in sorted(routes.items(), key=lambda kv: path_cost(kv[1])):
    print(f"{name}: {path_cost(p):7.2f} km   {' -> '.join(p)}")

assert path == routes["Route 1"]
assert abs(cost - min(path_cost(p) for p in routes.values())) < 1e-9
print("\nUCS result matches the cheapest Part 1 route.")

## 5. Conclusion

Uniform Cost Search expands towns in order of increasing path cost and returns the optimal route for the trailer:

**Nairobi → Naivasha → Gilgil → Nakuru → Kericho → Kisumu, 294.80 km.**

This is the cheapest route in the simplified network, and it matches Route 1 from Part 1.

## Group work evidence

*Insert the group photo (all members' faces visible) or minutes of the discussion here, or upload it as a separate file to the GitHub repository.*